# 12-05 Python 手撕代码

**高频考点**: 装饰器、异步编程、数据结构、设计模式、文本处理

---

In [ ]:
# Q1: 实现 LRU Cache
print("""
Q: 手写一个 LRU (Least Recently Used) Cache

要求:
  - get(key): O(1) 获取值，若不存在返回 -1
  - put(key, value): O(1) 插入/更新，超出容量时淘汰最久未使用的
  - 不能用 functools.lru_cache

思路:
  - OrderedDict: move_to_end() + popitem(last=False)
  - 或者: 哈希表 + 双向链表 (面试常考手写链表版)
""")

from collections import OrderedDict

class LRUCache:
    """基于 OrderedDict 的 LRU Cache"""
    
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = OrderedDict()
    
    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        # 访问后移到末尾 (最近使用)
        self.cache.move_to_end(key)
        return self.cache[key]
    
    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            # 弹出最前面的 (最久未使用)
            self.cache.popitem(last=False)

# 测试
lru = LRUCache(2)
lru.put(1, 1)
lru.put(2, 2)
print(f"get(1) = {lru.get(1)}")   # 1
lru.put(3, 3)                      # 淘汰 key=2
print(f"get(2) = {lru.get(2)}")   # -1 (已淘汰)
print(f"get(3) = {lru.get(3)}")   # 3

print("\n--- 进阶: 双向链表版 (面试手写) ---")

class Node:
    def __init__(self, key=0, val=0):
        self.key = key
        self.val = val
        self.prev = None
        self.next = None

class LRUCacheLinkedList:
    """基于哈希表 + 双向链表的 LRU Cache"""
    
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}  # key -> Node
        # 哨兵节点: head <-> ... <-> tail
        self.head = Node()
        self.tail = Node()
        self.head.next = self.tail
        self.tail.prev = self.head
    
    def _remove(self, node: Node):
        """从链表中删除节点"""
        node.prev.next = node.next
        node.next.prev = node.prev
    
    def _add_to_end(self, node: Node):
        """在 tail 之前插入节点 (最近使用)"""
        node.prev = self.tail.prev
        node.next = self.tail
        self.tail.prev.next = node
        self.tail.prev = node
    
    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        node = self.cache[key]
        self._remove(node)
        self._add_to_end(node)
        return node.val
    
    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            self._remove(self.cache[key])
        node = Node(key, value)
        self.cache[key] = node
        self._add_to_end(node)
        if len(self.cache) > self.capacity:
            # 删除 head 后面的节点 (最久未使用)
            lru_node = self.head.next
            self._remove(lru_node)
            del self.cache[lru_node.key]

# 测试链表版
lru2 = LRUCacheLinkedList(2)
lru2.put(1, 1)
lru2.put(2, 2)
print(f"get(1) = {lru2.get(1)}")   # 1
lru2.put(3, 3)                      # 淘汰 key=2
print(f"get(2) = {lru2.get(2)}")   # -1
print(f"get(3) = {lru2.get(3)}")   # 3
print("✓ 双向链表版通过")

In [ ]:
# Q2: 带指数退避的重试装饰器
print("""
Q: 实现一个 retry 装饰器，支持:
   - 最大重试次数
   - 指数退避 (exponential backoff)
   - 指定捕获的异常类型
   - 日志输出

场景: 调用 LLM API 时网络不稳定，需要自动重试
""")

import time
import functools
import random

def retry(max_retries=3, base_delay=1.0, exceptions=(Exception,)):
    """带指数退避的重试装饰器"""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_retries:
                        print(f"  [FAIL] 已达最大重试次数 {max_retries}")
                        raise
                    # 指数退避 + 随机抖动
                    delay = base_delay * (2 ** attempt) + random.uniform(0, 0.1)
                    print(f"  [RETRY] 第 {attempt+1} 次失败: {e}, "
                          f"{delay:.2f}s 后重试...")
                    time.sleep(delay)
        return wrapper
    return decorator

# 测试: 模拟不稳定的 API
call_count = 0

@retry(max_retries=3, base_delay=0.01, exceptions=(ConnectionError,))
def unstable_api_call():
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError(f"连接超时 (第{call_count}次)")
    return {"status": "ok", "data": "B站广告数据"}

call_count = 0
result = unstable_api_call()
print(f"结果: {result}")
print(f"总调用次数: {call_count}")

print("\n--- 进阶: async 版本 ---")

import asyncio

def async_retry(max_retries=3, base_delay=1.0, exceptions=(Exception,)):
    """异步版重试装饰器"""
    def decorator(func):
        @functools.wraps(func)
        async def wrapper(*args, **kwargs):
            for attempt in range(max_retries + 1):
                try:
                    return await func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_retries:
                        raise
                    delay = base_delay * (2 ** attempt)
                    print(f"  [ASYNC RETRY] 第 {attempt+1} 次失败, "
                          f"{delay:.2f}s 后重试...")
                    await asyncio.sleep(delay)
        return wrapper
    return decorator

@async_retry(max_retries=2, base_delay=0.01, exceptions=(TimeoutError,))
async def async_llm_call():
    global call_count
    call_count += 1
    if call_count < 2:
        raise TimeoutError("LLM 响应超时")
    return "异步调用成功"

call_count = 0
result = await async_llm_call()
print(f"Async 结果: {result}")

In [ ]:
# Q3: 异步生产者-消费者模式
print("""
Q: 用 asyncio 实现生产者-消费者模式

场景: Agent 系统中，生产者从消息队列取任务，消费者(多个 LLM worker)并发处理
要求:
  - asyncio.Queue 作为缓冲区
  - 多个消费者并发消费
  - 优雅退出 (poison pill 或 sentinel)
""")

import asyncio
import random

async def producer(queue: asyncio.Queue, name: str, num_tasks: int):
    """生产者: 生成广告审核任务"""
    for i in range(num_tasks):
        task = {"id": f"{name}-{i}", "content": f"广告素材_{i}", "priority": random.randint(1, 5)}
        await queue.put(task)
        print(f"  [生产] {name} → 任务 {task['id']} (优先级 {task['priority']})")
        await asyncio.sleep(random.uniform(0.01, 0.05))
    print(f"  [生产] {name} 完成")

async def consumer(queue: asyncio.Queue, name: str):
    """消费者: LLM Worker 处理任务"""
    processed = 0
    while True:
        task = await queue.get()
        if task is None:  # Poison pill → 退出
            queue.task_done()
            break
        # 模拟 LLM 处理
        await asyncio.sleep(random.uniform(0.02, 0.08))
        print(f"  [消费] {name} ← 任务 {task['id']} ✓")
        processed += 1
        queue.task_done()
    print(f"  [消费] {name} 退出, 共处理 {processed} 个任务")
    return processed

async def main():
    queue = asyncio.Queue(maxsize=5)  # 有界队列，背压控制
    
    num_consumers = 3
    num_tasks_per_producer = 4
    
    # 启动消费者
    consumers = [asyncio.create_task(consumer(queue, f"Worker-{i}"))
                 for i in range(num_consumers)]
    
    # 启动生产者
    producers = [
        asyncio.create_task(producer(queue, "审核队列", num_tasks_per_producer)),
    ]
    
    # 等待生产者完成
    await asyncio.gather(*producers)
    
    # 发送 poison pill 通知消费者退出
    for _ in range(num_consumers):
        await queue.put(None)
    
    # 等待消费者完成
    results = await asyncio.gather(*consumers)
    total = sum(results)
    print(f"\n总处理任务: {total}")

await main()

In [ ]:
# Q4: 并发 API 调用 + Semaphore 限流
print("""
Q: 用 asyncio.Semaphore 实现并发 API 调用限流

场景: 批量调用 LLM API 生成广告文案，API 限制最多 5 个并发
要求:
  - asyncio.gather 并发执行
  - Semaphore 控制最大并发数
  - 收集所有结果 + 错误处理
""")

import asyncio
import random
import time

async def call_llm_api(semaphore: asyncio.Semaphore, task_id: int):
    async with semaphore:
        print(f"  [开始] 任务 {task_id}")
        await asyncio.sleep(random.uniform(0.05, 0.15))
        if random.random() < 0.1:
            raise RuntimeError(f"任务 {task_id} API 超时")
        result = f"广告文案_{task_id}: B站年轻人的创意平台"
        print(f"  [完成] 任务 {task_id}")
        return result

async def batch_generate(task_ids: list, max_concurrent: int = 3):
    """批量并发生成，带限流和错误处理 (return_exceptions 保证全部完成)"""
    semaphore = asyncio.Semaphore(max_concurrent)

    start = time.time()
    raw = await asyncio.gather(
        *[call_llm_api(semaphore, tid) for tid in task_ids],
        return_exceptions=True,
    )
    elapsed = time.time() - start

    results = [
        {"id": tid, "status": "error", "error": str(r)} if isinstance(r, Exception)
        else {"id": tid, "status": "ok", "result": r}
        for tid, r in zip(task_ids, raw)
    ]
    success = [r for r in results if r["status"] == "ok"]
    failed = [r for r in results if r["status"] == "error"]

    print(f"\n--- 批量结果 ---")
    print(f"总任务: {len(results)}, 成功: {len(success)}, 失败: {len(failed)}")
    print(f"总耗时: {elapsed:.2f}s (并发={max_concurrent})")
    if failed:
        print(f"失败任务: {[r['id'] for r in failed]}")
    return results

# 测试: 10 个任务，最大并发 3
random.seed(42)
results = await batch_generate(list(range(10)), max_concurrent=3)

print("\n--- 对比: 不同并发数的耗时 ---")
# 使用 return_exceptions=True 保证所有任务都执行完，测量才有意义
for concurrency in [1, 3, 5, 10]:
    random.seed(42)
    start = time.time()
    semaphore = asyncio.Semaphore(concurrency)
    tasks = [call_llm_api(semaphore, i) for i in range(10)]
    await asyncio.gather(*tasks, return_exceptions=True)
    elapsed = time.time() - start
    print(f"  并发={concurrency:2d} → 耗时 {elapsed:.3f}s")

In [ ]:
# Q5: 日志分析 - 统计 Agent 调用链路
print("""
Q: 解析 Agent 调用日志，统计各 Tool 的调用次数、平均耗时、错误率

场景: Agent 系统上线后需要分析性能瓶颈
输入: 日志文本 (每行一条)
输出: 统计报表
""")

import re
from collections import defaultdict

LOG_DATA = """2024-01-15 10:00:01 INFO agent.tool_call tool=search status=success latency_ms=120
2024-01-15 10:00:02 INFO agent.tool_call tool=search status=success latency_ms=95
2024-01-15 10:00:03 ERROR agent.tool_call tool=search status=error latency_ms=5000
2024-01-15 10:00:04 INFO agent.tool_call tool=sql_query status=success latency_ms=230
2024-01-15 10:00:05 INFO agent.tool_call tool=sql_query status=success latency_ms=180
2024-01-15 10:00:06 INFO agent.tool_call tool=sql_query status=success latency_ms=210
2024-01-15 10:00:07 INFO agent.tool_call tool=sql_query status=error latency_ms=3000
2024-01-15 10:00:08 INFO agent.tool_call tool=llm_generate status=success latency_ms=1500
2024-01-15 10:00:09 INFO agent.tool_call tool=llm_generate status=success latency_ms=2200
2024-01-15 10:00:10 INFO agent.tool_call tool=llm_generate status=success latency_ms=1800
2024-01-15 10:00:11 INFO agent.tool_call tool=llm_generate status=success latency_ms=1600
2024-01-15 10:00:12 ERROR agent.tool_call tool=llm_generate status=error latency_ms=30000
2024-01-15 10:00:13 INFO agent.tool_call tool=vector_search status=success latency_ms=45
2024-01-15 10:00:14 INFO agent.tool_call tool=vector_search status=success latency_ms=38
2024-01-15 10:00:15 INFO agent.tool_call tool=vector_search status=success latency_ms=52"""

LOG_PATTERN = re.compile(
    r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) \w+ agent\.tool_call '
    r'tool=(\w+) status=(\w+) latency_ms=(\d+)'
)

def parse_log_line(line: str) -> dict | None:
    match = LOG_PATTERN.match(line)
    if not match:
        return None
    return {
        "timestamp": match.group(1),
        "tool": match.group(2),
        "status": match.group(3),
        "latency_ms": int(match.group(4)),
    }

def percentile(sorted_values: list, p: float) -> float:
    """取第 p 百分位 (p in [0, 1])，nearest-rank 方法"""
    idx = min(int(len(sorted_values) * p), len(sorted_values) - 1)
    return sorted_values[idx]

def analyze_logs(log_text: str) -> dict:
    """分析日志，生成结构化统计报表 (数值保持 float/int 类型)"""
    stats = defaultdict(lambda: {"success": 0, "error": 0, "latencies": []})

    for line in log_text.strip().split("\n"):
        parsed = parse_log_line(line)
        if not parsed:
            continue
        s = stats[parsed["tool"]]
        s[parsed["status"]] += 1
        s["latencies"].append(parsed["latency_ms"])

    report = {}
    for tool, data in stats.items():
        latencies = sorted(data["latencies"])
        total = data["success"] + data["error"]
        report[tool] = {
            "total_calls": total,
            "success_rate": data["success"] / total,
            "error_rate": data["error"] / total,
            "avg_latency_ms": sum(latencies) / len(latencies),
            "p50_latency_ms": percentile(latencies, 0.50),
            "p99_latency_ms": percentile(latencies, 0.99),
        }
    return report

report = analyze_logs(LOG_DATA)

print(f"{'Tool':<16} {'Calls':>6} {'Success%':>9} {'Avg(ms)':>8} {'P50(ms)':>8} {'P99(ms)':>8}")
print("-" * 60)
for tool, m in sorted(report.items()):
    print(f"{tool:<16} {m['total_calls']:>6} {m['success_rate']*100:>8.1f}% "
          f"{m['avg_latency_ms']:>8.0f} {m['p50_latency_ms']:>8} {m['p99_latency_ms']:>8}")

print("\n--- 发现性能瓶颈 ---")
slowest = max(report.items(), key=lambda x: x[1]["avg_latency_ms"])
print(f"最慢 Tool: {slowest[0]} (avg {slowest[1]['avg_latency_ms']:.0f}ms)")

highest_err = max(report.items(), key=lambda x: x[1]["error_rate"])
print(f"最高错误率: {highest_err[0]} ({highest_err[1]['error_rate']*100:.1f}%)")

In [ ]:
# Q6: 实现简易 Token 计数器 + 滑动窗口对话管理
print("""
Q: 实现一个对话历史管理器

场景: Agent 的 context window 有限 (如 4096 tokens)
要求:
  - 近似 token 计数 (中文≈2 token/字, 英文≈1 token/4字符)
  - 滑动窗口: 超出限制时从最早的消息开始删
  - 保留 system prompt 不被删
""")

class TokenCounter:
    """近似 Token 计数器"""
    
    @staticmethod
    def count(text: str) -> int:
        """中文字符≈2 token, 英文≈0.25 token/char"""
        chinese = sum(1 for c in text if '\u4e00' <= c <= '\u9fff')
        other = len(text) - chinese
        return int(chinese * 2 + other * 0.25) + 1  # 多算 1 token 作为 safety margin

class ConversationManager:
    """带 Token 限制的对话管理器"""
    
    def __init__(self, max_tokens: int = 4096, system_prompt: str = ""):
        self.max_tokens = max_tokens
        self.system_prompt = system_prompt
        self.system_tokens = TokenCounter.count(system_prompt)
        self.messages = []  # list of {"role": str, "content": str, "tokens": int}
    
    def add_message(self, role: str, content: str):
        """添加消息，超出限制时自动淘汰旧消息"""
        tokens = TokenCounter.count(content)
        self.messages.append({"role": role, "content": content, "tokens": tokens})
        
        while self._total_tokens() > self.max_tokens and len(self.messages) > 1:
            removed = self.messages.pop(0)
            print(f"  [淘汰] {removed['role']}: {removed['content'][:20]}... ({removed['tokens']} tokens)")
    
    def _total_tokens(self) -> int:
        return self.system_tokens + sum(m["tokens"] for m in self.messages)
    
    def get_context(self) -> list:
        """获取当前上下文 (system + messages)"""
        context = [{"role": "system", "content": self.system_prompt}]
        context.extend({"role": m["role"], "content": m["content"]} for m in self.messages)
        return context
    
    def stats(self):
        print(f"  消息数: {len(self.messages)}, "
              f"Token 用量: {self._total_tokens()}/{self.max_tokens} "
              f"({self._total_tokens()/self.max_tokens*100:.0f}%)")

# 测试
manager = ConversationManager(
    max_tokens=200,  # 故意设小，方便观察淘汰
    system_prompt="你是B站广告助手，帮助广告主优化投放策略。"
)

conversations = [
    ("user", "我想投放一个游戏广告，预算五万元，目标用户是18-25岁的男性。"),
    ("assistant", "好的，我建议您选择B站游戏区信息流广告，目标人群定向为18-25岁男性用户。"),
    ("user", "CTR一般能到多少？有什么优化建议？"),
    ("assistant", "游戏类广告平均CTR约2-3%，优化建议：1)使用游戏实录素材 2)标题突出福利 3)投放时间选在晚8-11点。"),
    ("user", "帮我制定一个详细的投放计划，包括每日预算分配和素材策略。"),
]

for role, content in conversations:
    print(f"\n+ [{role}] {content[:30]}...")
    manager.add_message(role, content)
    manager.stats()

print(f"\n最终上下文消息数: {len(manager.get_context())}")

In [ ]:
# Q7: 带 TTL (过期时间) 的函数缓存装饰器
print("""
Q: 实现带 TTL (过期时间) 的函数缓存装饰器

场景: RAG 系统中缓存 Embedding 结果，避免重复计算
要求:
  - 缓存函数结果
  - 支持 TTL 过期
  - 支持手动清除
""")

import time
from functools import wraps

def ttl_cache(ttl_seconds: int = 60):
    """带 TTL 的缓存装饰器"""
    def decorator(func):
        cache = {}  # key -> (value, expire_time)

        @wraps(func)
        def wrapper(*args, **kwargs):
            key = (args, tuple(sorted(kwargs.items())))
            now = time.time()

            if key in cache:
                value, expire_time = cache[key]
                if now < expire_time:
                    print(f"  [CACHE HIT] {func.__name__}{args}")
                    return value
                del cache[key]

            print(f"  [CACHE MISS] {func.__name__}{args}")
            result = func(*args, **kwargs)
            cache[key] = (result, now + ttl_seconds)
            return result

        wrapper.cache_clear = lambda: cache.clear()
        wrapper.cache_info = lambda: {"size": len(cache), "keys": list(cache.keys())}
        return wrapper
    return decorator

@ttl_cache(ttl_seconds=1)
def compute_embedding(text: str) -> list:
    """模拟 Embedding 计算 (耗时操作)"""
    time.sleep(0.1)
    return [hash(text) % 100 / 100.0 for _ in range(4)]

print("第1次调用 (miss):")
v1 = compute_embedding("B站广告投放")

print("第2次调用 (hit):")
v2 = compute_embedding("B站广告投放")
assert v1 == v2

print("不同参数 (miss):")
v3 = compute_embedding("游戏推广")

print(f"缓存大小: {compute_embedding.cache_info()['size']}")

print("等待 TTL 过期...")
time.sleep(1.1)

print("过期后调用 (miss):")
v4 = compute_embedding("B站广告投放")
print("TTL 缓存测试通过")

In [ ]:
# Q8: 实现简易 Pipeline (链式调用)
print("""
Q: 实现一个函数 Pipeline，支持链式调用

场景: RAG/Agent 中的数据处理管道
  text → clean → chunk → embed → store
要求:
  - 支持 pipe() 链式添加步骤
  - 支持 execute() 执行
  - 每步可以是任意 callable
""")

import re
from typing import Callable, Any

class Pipeline:
    """函数式 Pipeline"""

    def __init__(self):
        self.steps: list[tuple[str, Callable]] = []

    def pipe(self, name: str, func: Callable) -> "Pipeline":
        """添加步骤，返回 self 以支持链式调用"""
        self.steps.append((name, func))
        return self

    def execute(self, data: Any, verbose: bool = True) -> Any:
        """顺序执行所有步骤"""
        result = data
        for i, (name, func) in enumerate(self.steps):
            result = func(result)
            if verbose:
                preview = str(result)[:60]
                if len(str(result)) > 60:
                    preview += "..."
                print(f"  Step {i+1}/{len(self.steps)} [{name}]: {preview}")
        return result

WHITESPACE_RE = re.compile(r'\s+')

def clean_text(text: str) -> str:
    """清洗文本: 合并连续空白"""
    return WHITESPACE_RE.sub(' ', text.strip())

def chunk_text(text: str, chunk_size: int = 50) -> list[str]:
    """按字符数分块"""
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

def embed_chunks(chunks: list[str]) -> list[dict]:
    """模拟 Embedding"""
    return [{"text": c, "vector": [hash(c) % 100 / 100.0] * 4} for c in chunks]

def add_metadata(items: list[dict]) -> list[dict]:
    for i, item in enumerate(items):
        item["id"] = f"chunk_{i}"
        item["source"] = "bilibili_ad_docs"
    return items

raw_text = """
B站商业化广告平台    支持多种广告形式：信息流广告、开屏广告、 
搜索广告、UP主商单。广告主可通过自助投放平台进行精准人群定向，
支持年龄、性别、兴趣、地域等多维度定向。   计费模式包括 CPM、CPC、OCPC。
"""

print("=== RAG 文档处理 Pipeline ===\n")

pipeline = (Pipeline()
    .pipe("clean", clean_text)
    .pipe("chunk", lambda t: chunk_text(t, 40))
    .pipe("embed", embed_chunks)
    .pipe("metadata", add_metadata)
)

results = pipeline.execute(raw_text)
print(f"\n最终产出: {len(results)} 个带向量的文档块")
for r in results:
    print(f"  {r['id']}: text='{r['text'][:25]}...' vector_dim={len(r['vector'])}")

## 面试速查卡片

| 题目 | 核心考点 | 关键 API/模式 |
|------|---------|-------------|
| LRU Cache | 数据结构设计 | `OrderedDict.move_to_end()` / 双向链表+哈希表 |
| Retry 装饰器 | 装饰器 + 异常处理 | `functools.wraps`, 指数退避 `2**attempt`, `random` 抖动 |
| 生产者-消费者 | 异步编程 | `asyncio.Queue`, `create_task`, poison pill 退出 |
| 并发限流 | Semaphore | `asyncio.Semaphore`, `asyncio.gather`, 错误收集 |
| 日志分析 | 正则 + 聚合统计 | `re.match`, `defaultdict`, P50/P99 计算 |
| 对话管理器 | Token 管理 | 滑动窗口淘汰, 近似 token 计数, system prompt 保护 |
| TTL 缓存 | 装饰器 + 过期策略 | 闭包存 cache dict, `time.time()` 判断过期 |
| Pipeline | 链式调用 | `return self` 实现链式, `Callable` 类型, 函数组合 |

### 面试常见追问

| 追问 | 回答要点 |
|------|---------|
| LRU 为什么用双向链表? | 删除节点 O(1)，单链表需要 O(n) 找前驱 |
| retry 为什么要随机抖动? | 防止雷群效应 (thundering herd)，多个客户端同时重试压垮服务 |
| asyncio.gather vs TaskGroup? | Python 3.11+ 推荐 TaskGroup，异常处理更好 (一个失败全部取消) |
| Semaphore vs 连接池? | Semaphore 更通用 (不绑定资源); 连接池管理具体连接的复用和健康检查 |
| 为什么不用 functools.lru_cache? | 无 TTL、无法缓存不可哈希参数、无法分布式共享; 生产用 Redis 缓存 |